In [1]:
# Step 1: Install required packages
!pip install -U \
  langchain \
  langchain-community \
  langchain-text-splitters \
  faiss-cpu \
  sentence-transformers \
  pypdf \
  transformers \
  accelerate \
  huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 12.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.12
    Uninstalling langchain-1.2.12:
      Successfully uninstalled langchain-1.2.12


In [2]:
# Step 2: Import PDF loader + text splitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

In [3]:
# Step 3: Create embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_18063/1488283946.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Step 4: Load your PDF
pdf_path = "/content/sample_data/AI_in_modern_society.pdf"  # upload your PDF in Colab

loader = PyPDFLoader(pdf_path)
documents = loader.load()

In [5]:
# Step 5: Split text into smaller chunks size: 500 characters and overlap: 50 characters
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)

In [6]:
# Step 6: Create FAISS db and store embeddings
vector_db = FAISS.from_documents(docs, embedding_model)

In [7]:
# Step 7: Query + retrieve  (Similarity Search)
query = input("Enter your query: ")

results = vector_db.similarity_search(query, k=3)

Enter your query: Compare the benefits of AI in finance and healthcare


In [8]:
# Step 8 (OPTIONAL): Show retrieved results (matching chunks)
print("\nTop relevant chunks:\n")

for i, res in enumerate(results):
    print(f"Result {i+1}:")
    print(res.page_content)
    print("-" * 50)


Top relevant chunks:

Result 1:
Applications of AI 
AI has numerous real-world applications: 
Healthcare: 
AI assists in diagnosing diseases, analyzing medical images, and predicting 
patient outcomes. It helps doctors make faster and more accurate 
decisions. 
Finance: 
AI is used for fraud detection, algorithmic trading, and risk assessment. It 
can process vast amounts of ﬁnancial data quickly.
--------------------------------------------------
Result 2:
Transportation: 
Self-driving cars use AI to navigate roads, detect obstacles, and make real-
time decisions. 
Retail: 
AI improves customer experience through recommendation systems and 
chatbots. 
Education: 
AI enables personalized learning by adapting content based on student 
performance. 
 
Beneﬁts of AI 
AI oƯers several advantages: 
 EƯiciency: Automates repetitive tasks, saving time and resources 
 Accuracy: Reduces human error in data processing
--------------------------------------------------
Result 3:
Artiﬁcial Inte

In [9]:
# Step 9: Add Hugging Face LLM

# Load text generation pipeline
# Uses the HuggingFace Transformers library
# Downloads the model google/flan-t5-base from Hugging Face Hub
# Runs it locally in your Colab runtime
pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",   # lightweight + good for QA
    max_new_tokens=256
)

# Wraps the Hugging Face pipeline into LangChain format
# So it can work with your RAG pipeline
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM

In [10]:
# Step 10: Generate Final answer for the user

context = "\n\n".join([doc.page_content for doc in results])

prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print("\nFinal Answer:\n")
print(response)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Final Answer:


Answer the question based only on the context below.

Context:
Applications of AI 
AI has numerous real-world applications: 
Healthcare: 
AI assists in diagnosing diseases, analyzing medical images, and predicting 
patient outcomes. It helps doctors make faster and more accurate 
decisions. 
Finance: 
AI is used for fraud detection, algorithmic trading, and risk assessment. It 
can process vast amounts of ﬁnancial data quickly.

Transportation: 
Self-driving cars use AI to navigate roads, detect obstacles, and make real-
time decisions. 
Retail: 
AI improves customer experience through recommendation systems and 
chatbots. 
Education: 
AI enables personalized learning by adapting content based on student 
performance. 
 
Beneﬁts of AI 
AI oƯers several advantages: 
 EƯiciency: Automates repetitive tasks, saving time and resources 
 Accuracy: Reduces human error in data processing

Artiﬁcial Intelligence in Modern Society 
 
Introduction to Artiﬁcial Intelligence 
Art